# AntMaze-v1 — train extra paper-faithful seeds (252 / 173)

Complements the existing `antmaze_s221_paper` checkpoint with **additional training seeds** so the planner-comparison ablation can report results across ≥2–3 training seeds (the main limitation flagged in `docs/EXPERIMENTAL_DESIGN_planners.md`). Entry point **`rl.main_latent`** (launcher `launcher_latent`), same as the paper's `scripts/antmaze.sh`.

**Usage:** set `SEED=252` in the train cell, run to convergence, **Save Version**; then set `SEED=173` and repeat (a separate output). Structural flags are paper-faithful (from the eval `ENV_CFG` / `antmaze.sh`); the **training-loop** flags (`n_cycles`, `n_initial_rollouts`, `n_epochs`) are sensible defaults — the notebook `cat`s `scripts/antmaze.sh` so you can align them if the paper differs. Watch `Test_TestEnv_PlanSuccessRate` and stop when it plateaus (~0.8, like s221). AntMaze training is long → **resume-aware** across sessions.

## 1. Code + MuJoCo env (~10–15 min first time)

In [ ]:
import os
if os.path.isdir('/kaggle/working/latent_landmarks'):
    !cd /kaggle/working/latent_landmarks && git pull -q origin retrain
else:
    !git clone -q -b retrain https://github.com/Jun1801/latent_landmarks.git /kaggle/working/latent_landmarks
if not os.path.isdir('/kaggle/working/wmag'):
    !git clone -q https://github.com/LunjunZhang/world-model-as-a-graph /kaggle/working/wmag
!bash /kaggle/working/latent_landmarks/repro/setup_kaggle.sh

## 2. (Resume only) Restore prior checkpoint — weights + optimizer + **replay**
Skip for a fresh run. Resuming training needs the full state incl. `replay_0.pt` (large). Set `SEED` to match the run you are continuing; find `<SLUG>` with `!ls /kaggle/input/`.

In [ ]:
!ls /kaggle/input/
import os, shutil
SEED = 252                              # <-- 252 or 173
SLUG = 'PUT-DATASET-SLUG-HERE'          # <-- output that saved antmaze_s{SEED}
CKPT, ENV = f'antmaze_s{SEED}', 'AntMaze-v1'
src = f'/kaggle/input/{SLUG}/experiments/{ENV}/{CKPT}/state'
dst = f'/kaggle/working/experiments/{ENV}/{CKPT}/state'; os.makedirs(dst, exist_ok=True)
if os.path.isdir(src):
    for f in os.listdir(src):
        shutil.copy(f'{src}/{f}', f'{dst}/{f}')
    print('restored ->', sorted(os.listdir(dst)))
else:
    print('no prior state at', src, '-> fresh run')

## 3. Verify GPU + MuJoCo + env; show the paper's antmaze.sh for flag reference

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-}; cd /kaggle/working/wmag && conda run -n l3p python -c "import torch, mujoco_py, gym; print('cuda', torch.cuda.is_available()); e=gym.make('AntMaze-v1'); print('env ok', e.observation_space['observation'].shape, e.action_space.shape)"
!echo "===== wmag/scripts/antmaze.sh (reference — align n_cycles/n_epochs/etc if needed) =====" && cat /kaggle/working/wmag/scripts/antmaze.sh 2>/dev/null || echo "(antmaze.sh not found — using ENV_CFG structural flags below)"

## 4. Train — resume-aware (edit `SEED`)
`--n_epochs 500` is a cap, not a target — **stop when `Test_TestEnv_PlanSuccessRate` plateaus**. Re-running auto-adds `--resume_ckpt` if a checkpoint exists (restores weights/optimizer/replay). `--n_workers 3` fits Kaggle's ~4 CPUs. Structural flags copied verbatim from the paper-faithful eval `ENV_CFG`; cross-check the loop flags against the `antmaze.sh` printed above.

In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-}
cd /kaggle/working/wmag
SEED=252                       # <-- 252 or 173 (match cell 2 if resuming)
SAVE=/kaggle/working/experiments
CKPT=antmaze_s$SEED
RESUME=""
if [ -f "$SAVE/AntMaze-v1/$CKPT/state/algo.pt" ]; then
  RESUME="--resume_ckpt $CKPT"; echo "[resume] continuing $CKPT"
else
  echo "[fresh] new run $CKPT"
fi
conda run -n l3p python -m rl.main_latent \
  --env_name AntMaze-v1 --test_env_name AntMazeTest-v1 --cuda \
  --seed $SEED --n_workers 3 \
  --gamma 0.98 --clip_return 100 --future_step 100 \
  --n_latent_landmarks 50 --n_extra_landmark 150 --dist_clip -20.0 \
  --latent_batch_size 256 --batch_size 1000 \
  --grad_value_clipping -1.0 --grad_norm_clipping 15.0 \
  --action_l2 0.05 --optimize_every 2 \
  --n_cycles 15 --n_initial_rollouts 100 --n_test_rollouts 10 \
  --n_epochs 500 \
  --ckpt_name $CKPT --save_dir $SAVE \
  $RESUME

## 5. Persist + next step
**Save Version** so `/kaggle/working/experiments/AntMaze-v1/antmaze_s{SEED}/state/` lands in the output. Then in `antmaze_classical.ipynb` / `antmaze_mcts.ipynb` set the restore `CKPT` and `--resume_ckpt` to `antmaze_s252` (and `antmaze_s173`) to run the ablation on the new seeds → report mean over {221, 252, 173} with a training-seed CI. (Eval needs only `agent.pt`+`algo.pt`; resume-training needs `replay_0.pt` too.)

In [ ]:
!ls -la /kaggle/working/experiments/AntMaze-v1/*/state/ 2>/dev/null || echo 'no checkpoint yet'